# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
!pip install python-dotenv
!pip install dask

import os
from glob import glob
from dotenv import load_dotenv
load_dotenv()
pricadata = os.getenv('PRICEDATA')


In [2]:
!pip install ipykernel


In [3]:
!python -m ipykernel install --user --name dsi_py310 --display-name "Python 3.10 (DSI)"


Installed kernelspec dsi_py310 in C:\Users\amans\AppData\Roaming\jupyter\kernels\dsi_py310


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [4]:
!pip install pandas dask openpyxl
!pip install pyarrow
import pandas as pd

!pip install pandas
import dask.dataframe as dd
!pip install openpyxl
!pip install pandas dask openpyxl

!pip install --upgrade pyarrow

In [5]:
import os
from glob import glob
import dask.dataframe as dd

# Load environment variable (ensure you load your .env file earlier in your notebook)
PRICE_DATA = os.getenv("PRICE_DATA")
if PRICE_DATA is None:
    # Fallback if env variable is not set
    PRICE_DATA = r"C:\Users\amans\DSI WEEK 6\DSI Production\05_src\data\prices_csv"

# Find all parquet files in the directory and subdirectories
parquet_files = glob(os.path.join(PRICE_DATA, '**', '*.parquet'), recursive=True)
print(parquet_files)

# Load all parquet files as one Dask dataframe
ddf = dd.read_parquet(parquet_files)
print(ddf.head())


['C:/Users/amans/DSI WEEK 6/DSI Production/05_src/data/prices_csv\\ACN\\ACN_2001\\part.0.parquet', 'C:/Users/amans/DSI WEEK 6/DSI Production/05_src/data/prices_csv\\ACN\\ACN_2001\\part.1.parquet', 'C:/Users/amans/DSI WEEK 6/DSI Production/05_src/data/prices_csv\\ACN\\ACN_2002\\part.0.parquet', 'C:/Users/amans/DSI WEEK 6/DSI Production/05_src/data/prices_csv\\ACN\\ACN_2002\\part.1.parquet', 'C:/Users/amans/DSI WEEK 6/DSI Production/05_src/data/prices_csv\\ACN\\ACN_2003\\part.0.parquet', 'C:/Users/amans/DSI WEEK 6/DSI Production/05_src/data/prices_csv\\ACN\\ACN_2003\\part.1.parquet', 'C:/Users/amans/DSI WEEK 6/DSI Production/05_src/data/prices_csv\\ACN\\ACN_2004\\part.0.parquet', 'C:/Users/amans/DSI WEEK 6/DSI Production/05_src/data/prices_csv\\ACN\\ACN_2004\\part.1.parquet', 'C:/Users/amans/DSI WEEK 6/DSI Production/05_src/data/prices_csv\\ACN\\ACN_2005\\part.0.parquet', 'C:/Users/amans/DSI WEEK 6/DSI Production/05_src/data/prices_csv\\ACN\\ACN_2005\\part.1.parquet', 'C:/Users/amans/DSI

In [ ]:
# Install necessary packages (run in a notebook cell)
!pip install --upgrade numpy dask[complete] pyarrow

import os
from glob import glob
import dask.dataframe as dd

PRICE_DATA = os.getenv("PRICE_DATA")
if PRICE_DATA is None:
    PRICE_DATA = r"C:\Users\amans\DSI WEEK 6\DSI Production\05_src\data\prices_csv"

ddf = dd.read_parquet(PRICE_DATA)

ddf['Date'] = dd.to_datetime(ddf['Date'])
ddf = ddf.map_partitions(lambda df: df.sort_values(['ticker', 'Date']))

dd_feat = ddf.copy()

# Provide meta for shift to suppress warnings (dtype float64 assumed for Close columns)
dd_feat['Close_lag_1'] = dd_feat.groupby('ticker')['Close'].shift(1, meta=('Close', 'f8'))
dd_feat['Adj_Close_lag_1'] = dd_feat.groupby('ticker')['Adj Close'].shift(1, meta=('Adj Close', 'f8'))

dd_feat['returns'] = dd_feat['Close'] / dd_feat['Close_lag_1'] - 1
dd_feat['hi_lo_range'] = dd_feat['High'] - dd_feat['Low']

dd_feat.head(5, compute=True)


In [ ]:
print(ddf.columns)


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.

# for  the each ticker, calculate features
dd_feat = ddf.copy()

# one  day lag for close and adj Ccose
dd_feat['Close_lag_1'] = dd_feat.groupby('ticker')['Close'].shift(1, meta=('Close', 'f8'))
dd_feat['Adj_Close_lag_1'] = dd_feat.groupby('ticker')['Adj Close'].shift(1, meta=('Adj Close', 'f8'))

# returns
dd_feat['returns'] = dd_feat['Close'] / dd_feat['Close_lag_1'] - 1

# high-low range
dd_feat['hi_lo_range'] = dd_feat['High'] - dd_feat['Low']


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [25]:
# Write your code below.
# compute all Dask tasks & get as pandas dataframe 
df_pd = dd_feat.compute()

# make sure data is sorted for rolling by 'ticker' and 'Date'
df_pd = df_pd.sort_values(['ticker', 'Date'])

# add 10 day moving average of 'returns' for each ticker
df_pd['returns_ma10'] = df_pd.groupby('ticker')['returns'].rolling(10).mean().reset_index(level=0, drop=True)

# display first rows to verify
print(df_pd.head())



Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It was not necessary to convert to pandas; Dask can compute moving averages directly.
Using Dask is better for large datasets, as it avoids memory issues and scales more efficiently.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.